# Exercise 07 — Prompting: Solving Math Word Problems

In notebook 0 you changed the model's **weights** to teach it a task. This notebook leaves the weights alone and changes the **input** instead. Large instruction-tuned language models can do many tasks from a description alone (**zero-shot**) or from a description plus a few worked examples (**few-shot**). The examples are part of the prompt; the model learns from them *in context*, without a single gradient step. This was one of the main findings of the GPT-3 paper ([Brown et al., 2020](https://arxiv.org/abs/2005.14165)).

You will prompt a 27-billion-parameter open-weights model, served through the free tier of [Groq](https://groq.com), to solve **GSM8K** grade-school math word problems. The main question: how much does *the way you ask* change the accuracy? In particular, you compare asking for the answer directly with asking the model to write out its reasoning first, which is called **chain-of-thought** prompting ([Wei et al., 2022](https://arxiv.org/abs/2201.11903); [Kojima et al., 2022](https://arxiv.org/abs/2205.11916)).

## What you will do
1. **Setup**: an API key and your first request.
2. **The chat format**: roles, system prompts, conversations and temperature.
3. **GSM8K**: the dataset, and parsing its reference solutions.
4. **Getting a number out**: extracting the final answer from the model's reply.
5. **Direct answers**: zero-shot and few-shot.
6. **Chain-of-thought**: zero-shot and few-shot, and what it costs.
7. **Sampling and self-consistency**: many reasoning paths and a majority vote.
8. **Multiple-choice questions**: to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**1.1**, **1.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell that verifies your work automatically. Run it and make sure it passes before you move on.

> **Heads up: this notebook calls an online API.** You do not need a GPU, but you need an internet connection and a free Groq account (Part 1). The free tier is rate-limited. As of October 2026, the model used here allows 30 requests and 8,000 tokens per minute, and 1,000 requests and 200,000 tokens per day. The experiments are sized to use about half of the daily token budget. Because of the per-minute limit they take 10–15 minutes in total; the client waits and retries automatically when it hits the limit. Every reply is cached in a file `groq_cache.json` next to the notebook, so running a cell a second time costs nothing.

In [ ]:
%matplotlib inline
import hashlib
import json
import os
import re
from collections import Counter
from getpass import getpass

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset
from groq import Groq

## 1. Setup

Groq builds chips specialised for running language models (*LPUs*) and offers an API to models hosted on them. Like most providers, the API follows the format of OpenAI's *chat completions* API, so what you learn here carries over to other providers almost unchanged (see `COMPUTE_RESOURCES.md` for other free options).

**1.1 Get an API key.**
1. Create a free account at [console.groq.com](https://console.groq.com). The free tier needs no payment details.
2. Go to **API Keys** and create a key. Copy it: it is only shown once.
3. Run the cell below and paste the key when you are asked for it.

> **Keep your key secret.** Never write it into a code cell: anyone who sees the notebook (or the git repository you commit it to) could use your account. The cell below asks for the key without showing it. If you set the environment variable `GROQ_API_KEY` before starting Jupyter, it does not ask at all.

In [ ]:
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")

client = Groq(max_retries=8)  # reads GROQ_API_KEY; on a rate-limit error it waits and tries again (up to 8 times)

# A 27 B parameter open-weights model. If it is no longer available, pick another one from
# https://console.groq.com/docs/models and check its rate limits.
MODEL = "qwen/qwen3.8-27b"

All requests in this notebook go through the function `chat` below. It sends a conversation (a list of messages, explained in Part 2) to the model and returns the text of the reply and the number of tokens the request used: the prompt and the reply together, which is what the rate limits count.

Two details:
- Replies are **cached**: a request that was sent before is answered from `groq_cache.json` instead of the API. The argument `sample` lets you ask the very same question several times, which you need when sampling (Parts 2 and 7).
- `reasoning_effort="none"` switches off this model's built-in "thinking" mode, in which it reasons in hidden tokens before it answers. We want to control the reasoning ourselves, through the prompt.

In [ ]:
CACHE_PATH = "groq_cache.json"
try:
    with open(CACHE_PATH, encoding="utf-8") as f:
        cache = json.load(f)
except FileNotFoundError:
    cache = {}


def chat(messages, temperature=0.0, max_tokens=1024, sample=0):
    # Send a conversation to the model; returns the reply (a string) and the number of tokens used.
    key = hashlib.sha256(json.dumps([MODEL, messages, temperature, max_tokens, sample]).encode()).hexdigest()
    if key not in cache:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=temperature,
            max_completion_tokens=max_tokens,
            reasoning_effort="none",
        )
        cache[key] = {"reply": response.choices[0].message.content, "tokens": response.usage.total_tokens}
        with open(CACHE_PATH, "w", encoding="utf-8") as f:
            json.dump(cache, f)
    return cache[key]["reply"], cache[key]["tokens"]


reply, tokens = chat([{"role": "user", "content": "In one sentence: what is a language model?"}])
print(reply)
print(f"({tokens} tokens)")

## 2. The chat format

A request to a chat model is a list of **messages**. Each message is a dictionary with a `role` and a `content`:
- `"system"`: instructions that set up the whole conversation (who the model is, how it should answer). Optional, and placed first.
- `"user"`: what the user says.
- `"assistant"`: what the model said earlier in the conversation.

```python
[
    {"role": "system", "content": "You are a helpful math tutor."},
    {"role": "user", "content": "What is 2 + 2?"},
]
```

Behind the API, the messages are joined into a single text with special tokens that mark where each message begins and ends (the model's *chat template*). The model then continues that text as the assistant, exactly as your mini GPT continued Shakespeare. You will look at a chat template in notebook 2.

**2.1 Write a conversation** with a system message that makes the model answer every question in Danish and in at most 15 words, and a user message that asks why the sky is blue.

In [ ]:
messages = ...  # TODO: a list with a system message and a user message

reply, tokens = chat(messages)
print(reply)

In [ ]:
# ✅ Check your conversation
assert [m["role"] for m in messages] == ["system", "user"], "the conversation should be a system message followed by a user message"
assert all(isinstance(m["content"], str) and m["content"] for m in messages), "every message needs a non-empty content"
print("Looks good ✅")

### Conversations

The API is **stateless**: it does not remember earlier requests. To continue a conversation, you send the *whole* conversation again, with the model's earlier replies as `assistant` messages.

**2.2 Continue the conversation**: append the model's reply from 2.1 and a follow-up question, *"And why is it red at sunset?"*, and send everything again.

In [ ]:
follow_up = ...  # TODO: the conversation from 2.1, plus the reply as an assistant message, plus the follow-up question

reply_2, _ = chat(follow_up)
print(reply_2)

In [ ]:
# ✅ Check your follow-up
assert [m["role"] for m in follow_up] == ["system", "user", "assistant", "user"], "expected system, user, assistant, user"
assert follow_up[2]["content"] == reply, "the assistant message should be the model's reply from 2.1"
print("Looks good ✅")

Note that the `assistant` messages do not have to be real replies of the model: you can write them yourself. That is how **few-shot** prompts are built in Part 5. You show the model a few questions together with "its own" answers, and it continues in the same way.

### Temperature

As with your mini GPT last week, the model produces a probability distribution over the next token, and the reply is sampled from it. The `temperature` reshapes that distribution: at `0` the model (almost) always picks the most likely token, so the same prompt gives the same reply; higher values give more varied replies.

**2.3 Run the cell below**, which asks the same question three times at each temperature.

In [ ]:
prompt = [{"role": "user", "content": "Invent a name for a new coffee shop in Aarhus. Reply with the name only."}]
for temperature in [0.0, 1.0]:
    names = [chat(prompt, temperature=temperature, sample=i)[0] for i in range(3)]
    print(f"temperature {temperature}: {names}")

**2.4 What do you see?** For which kinds of tasks would you use a temperature of 0, and for which a higher one? Which one should we use to *measure* the accuracy on a math benchmark, and why?

---

*Your answer here:*

---

## 3. GSM8K

[GSM8K](https://huggingface.co/datasets/openai/gsm8k) (Grade School Math 8K; [Cobbe et al., 2021](https://arxiv.org/abs/2110.14168)) contains about 8,500 math word problems written by humans: 7,473 for training and 1,319 for testing. Each one takes between 2 and 8 steps of basic arithmetic. For years it was one of the standard benchmarks for the reasoning abilities of language models.

The dataset has two configurations, `main` and `socratic`; we use `main`.

In [ ]:
gsm8k = load_dataset("openai/gsm8k", "main")
print(gsm8k)

example = gsm8k["train"][0]
print("\nQUESTION:\n" + example["question"])
print("\nANSWER:\n" + example["answer"])

Every reference answer has the same structure:
- one or more lines of **reasoning**, one step per line,
- inside the reasoning, **calculator annotations** such as `<<48/2=24>>`. The annotators used them to have the arithmetic checked, and they are not meant to be read,
- a final line `#### <answer>` with the **final answer**, a number (sometimes written with thousands separators, such as `1,000`).

**3.1 Complete `parse_gsm8k`**, which splits a reference answer into the list of reasoning steps (without the calculator annotations) and the final answer as a float.

Hints:
- `re.sub(r"<<[^>]*>>", "", text)` removes all calculator annotations.
- `text.split("####")` separates the reasoning from the final answer.
- `str.splitlines()` splits a text into lines. Leave out lines that are empty.

In [ ]:
def parse_gsm8k(answer):
    # "Step 1\nStep 2 <<2*3=6>>6\n#### 1,000" -> (["Step 1", "Step 2 6"], 1000.0)
    steps = ...  # TODO: the reasoning lines, without calculator annotations and without empty lines
    final = ...  # TODO: the number after "####", as a float (remove thousands separators first)
    return steps, final


parse_gsm8k(example["answer"])

In [ ]:
# ✅ Check your parse_gsm8k
steps, final = parse_gsm8k(example["answer"])
assert steps == [
    "Natalia sold 48/2 = 24 clips in May.",
    "Natalia sold 48+24 = 72 clips altogether in April and May.",
], f"unexpected steps: {steps}"
assert final == 72.0 and isinstance(final, float), "the final answer should be the float 72.0"
assert parse_gsm8k("He pays 2*500 = <<2*500=1000>>1,000 dollars.\n\n#### 1,000") == (["He pays 2*500 = 1,000 dollars."], 1000.0), "remove empty lines and thousands separators"
print("Looks good ✅")

We turn every problem into a dictionary with the question, the reasoning steps and the final answer. From the training split we take a few **demonstrations** for the few-shot prompts; from the test split, the problems we evaluate on.

`N_TEST = 20` test problems is a small sample, chosen to stay within the free tier. With 20 problems, every problem is worth 5 percentage points, so treat small differences between methods with caution (see question 6.3).

In [ ]:
N_DEMOS = 5  # demonstrations in the few-shot prompts
N_TEST = 20  # test problems


def to_problem(row):
    steps, answer = parse_gsm8k(row["answer"])
    return {"question": row["question"], "steps": steps, "answer": answer}


demos = [to_problem(row) for row in gsm8k["train"].shuffle(seed=42).select(range(N_DEMOS))]
test_problems = [to_problem(row) for row in gsm8k["test"].shuffle(seed=42).select(range(N_TEST))]
print(json.dumps(demos[0], indent=2))

## 4. Getting a number out

To score the model automatically we need its final answer as a number, but the model replies with free text. We make this manageable by asking for a fixed format: the reply must end with a line `Answer: <number>`. Models do not always follow instructions to the letter, though. They write `**Answer:** 18`, `Answer: $1,080.`, or forget the line altogether. Robust parsing is a real part of every LLM evaluation.

(An alternative is to ask for JSON, and many APIs, Groq's included, have a *JSON mode* that forces the reply to be valid JSON. A simple answer line is enough for us.)

**4.1 Complete `extract_answer`.**
- If the reply contains `Answer:`, return the **first** number after its **last** occurrence.
- Otherwise, return the **last** number in the reply, the model's most likely final result.
- If there is no number at all, return `None`.

Hints:
- `NUMBER` below is a regular expression for numbers like `-3`, `1,080` and `2.5`; `re.findall(NUMBER, text)` returns all of them as strings.
- `text.rsplit("Answer:", 1)[-1]` is the text after the last `Answer:`.

In [ ]:
NUMBER = r"-?\d[\d,]*(?:\.\d+)?"


def extract_answer(reply):
    # The model's final answer as a float, or None if there is none
    ...  # TODO


def is_correct(predicted, gold):
    return predicted is not None and abs(predicted - gold) < 1e-6

In [ ]:
# ✅ Check your extract_answer
cases = {
    "Answer: 72": 72.0,
    "She sold 48 + 24 = 72 clips.\nAnswer: $1,080.": 1080.0,
    "**Answer:** 18 dollars": 18.0,
    "Answer: -2.5": -2.5,
    "First 3 + 4 = 7, then 7 * 2 = 14.": 14.0,
    "Answer: 12\nAnswer: 15": 15.0,
    "I cannot solve this.": None,
}
for reply, expected in cases.items():
    got = extract_answer(reply)
    assert got == expected, f"extract_answer({reply!r}) should be {expected}, got {got}"
print("Looks good ✅")

## 5. Direct answers

First we ask for the answer **directly**, without any explanation. The system prompts for both styles of prompting are below; the second one is for Part 6.

In a **few-shot** prompt, every demonstration is a `user` message with the demonstration's question, followed by an `assistant` message with the answer *in exactly the format we want*. For direct answers, the assistant message is only the answer line. For chain-of-thought (Part 6), it is the reasoning steps, one per line, followed by the answer line.

**5.1 Complete `build_messages`**: the system message, then a user and an assistant message per demonstration, then the question.

Hint: use `number_to_text` to write numbers, so that `72.0` appears as `72`.

In [ ]:
SYSTEM_DIRECT = (
    "You solve grade-school math word problems. Reply with only the final answer, "
    "as a single line of the form 'Answer: <number>'. Do not explain your reasoning."
)
SYSTEM_COT = (
    "You solve grade-school math word problems. Think step by step: write out your reasoning, "
    "one short step per line, and then give the final answer on a last line of the form 'Answer: <number>'."
)


def number_to_text(x):
    return str(int(x)) if x == int(x) else str(x)


def build_messages(question, demos, cot=False):
    # The prompt for one question: system message, demonstrations, question
    messages = ...  # TODO: start with the system message (SYSTEM_COT if cot else SYSTEM_DIRECT)
    for demo in demos:
        answer = ...  # TODO: "Answer: <number>", preceded by the reasoning steps (one per line) if cot
        ...           # TODO: add the demonstration as a user and an assistant message
    ...               # TODO: add the question as a user message
    return messages

In [ ]:
# ✅ Check your build_messages (this does not call the API)
m = build_messages("What is 2 + 3?", demos[:2])
assert [x["role"] for x in m] == ["system", "user", "assistant", "user", "assistant", "user"], "expected system, then user/assistant per demonstration, then user"
assert m[0]["content"] == SYSTEM_DIRECT and m[-1]["content"] == "What is 2 + 3?"
assert m[1]["content"] == demos[0]["question"], "the user message of a demonstration is its question"
assert m[2]["content"] == f"Answer: {number_to_text(demos[0]['answer'])}", "a direct demonstration answer is only 'Answer: <number>'"

m = build_messages("What is 2 + 3?", demos[:2], cot=True)
assert m[0]["content"] == SYSTEM_COT
assert m[2]["content"] == "\n".join(demos[0]["steps"]) + f"\nAnswer: {number_to_text(demos[0]['answer'])}", "a chain-of-thought demonstration is the steps, one per line, then the answer line"
assert len(build_messages("What is 2 + 3?", [])) == 2, "without demonstrations: only the system message and the question"
print("Looks good ✅")

Here is what a 2-shot prompt looks like, and the model's reply to one test question:

In [ ]:
for message in build_messages(test_problems[0]["question"], demos[:2]):
    print(f"--- {message['role']} ---\n{message['content']}\n")

reply, tokens = chat(build_messages(test_problems[0]["question"], demos[:2]))
print(f"=== reply ({tokens} tokens) ===\n{reply}")
print(f"\nextracted: {extract_answer(reply)}, correct answer: {test_problems[0]['answer']}")

### The experiment

`run_experiment` sends every test problem with the same kind of prompt, extracts the answers and collects everything in a table (a pandas `DataFrame`, one row per problem). It prints the accuracy and the average number of tokens per problem.

**5.2 Run the direct-answer experiments**, zero-shot and 5-shot. This takes a few minutes the first time, because of the rate limit.

In [ ]:
def run_experiment(name, demos, cot, problems=test_problems, **chat_kwargs):
    rows = []
    for i, problem in enumerate(problems):
        reply, tokens = chat(build_messages(problem["question"], demos, cot), **chat_kwargs)
        predicted = extract_answer(reply)
        rows.append({
            "setting": name, "problem": i, "gold": problem["answer"], "predicted": predicted,
            "correct": is_correct(predicted, problem["answer"]), "tokens": tokens, "reply": reply,
        })
    df = pd.DataFrame(rows)
    print(f"{name:<16} accuracy {df['correct'].mean():>4.0%}   {df['tokens'].mean():>5.0f} tokens per problem")
    return df


results = {}
results["direct, 0-shot"] = run_experiment("direct, 0-shot", [], cot=False)
results["direct, 5-shot"] = run_experiment("direct, 5-shot", demos, cot=False)

**5.3 How well does the model do when it has to answer directly?** Do the demonstrations help? Think about what the model can learn from five demonstrations that consist of a question and a bare number: does it learn to *do math*, or something else?

---

*Your answer here:*

---

## 6. Chain-of-thought

To answer directly, the model has to produce the result of a multi-step calculation as its *first* token(s), with a fixed amount of computation per token. **Chain-of-thought** (CoT) prompting lets the model write out intermediate steps first. Every step it writes becomes part of the input for the next tokens, so the model can build on its own intermediate results.

There are two variants:
- **Zero-shot CoT** ([Kojima et al., 2022](https://arxiv.org/abs/2205.11916)): just tell the model to think step by step; that is what `SYSTEM_COT` does.
- **Few-shot CoT** ([Wei et al., 2022](https://arxiv.org/abs/2201.11903)): demonstrations that include the worked reasoning.

**6.1 Run the chain-of-thought experiments.** Replies are now longer, so we allow more tokens per reply.

In [ ]:
results["CoT, 0-shot"] = run_experiment("CoT, 0-shot", [], cot=True, max_tokens=1024)
results["CoT, 5-shot"] = run_experiment("CoT, 5-shot", demos, cot=True, max_tokens=1024)

In [ ]:
summary = pd.DataFrame({
    name: {"accuracy": df["correct"].mean(), "tokens per problem": df["tokens"].mean()}
    for name, df in results.items()
}).T
print(summary)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
summary["accuracy"].plot.barh(ax=axes[0], xlim=(0, 1), title="accuracy")
summary["tokens per problem"].plot.barh(ax=axes[1], title="tokens per problem")
axes[1].set_yticklabels([])
plt.tight_layout()
plt.show()

**6.2 Compare the four settings.** How much does chain-of-thought help, and what does it cost? Why does writing out the steps help a model that generates one token at a time?

---

*Your answer here:*

---

**6.3 Look at the problems that the CoT prompts got wrong.** The cell below prints the reasoning for each of them.

In [ ]:
for name in ["CoT, 0-shot", "CoT, 5-shot"]:
    df = results[name]
    for _, row in df[~df["correct"]].iterrows():
        print(f"===== {name}, problem {row['problem']}: predicted {row['predicted']}, correct {row['gold']} =====")
        print(test_problems[row["problem"]]["question"])
        print("---")
        print(row["reply"])
        print()

Where do the errors come from: an arithmetic slip, misreading the question, a wrong plan, a parsing problem, or a mistake in the reference answer? With only 20 test problems, how confident are you that one setting is better than another?

---

*Your answer here:*

---

## 7. Sampling and self-consistency

At temperature 0 the model follows its single most likely reasoning path, and if that path goes wrong, the answer is wrong. **Self-consistency** ([Wang et al., 2023](https://arxiv.org/abs/2203.11171)) samples several reasoning paths at a higher temperature and takes a **majority vote** over their final answers. Correct reasoning paths tend to agree on the same answer, while wrong ones scatter over many different answers.

**7.1 Complete `majority_vote`**, which returns the most common answer, ignoring `None`s (or `None` if all answers are `None`).

Hint: `Counter(values).most_common(1)` returns a list with the most common value and its count, `[(value, count)]`.

In [ ]:
def majority_vote(answers):
    # The most common answer that is not None, or None if there is none
    ...  # TODO

In [ ]:
# ✅ Check your majority_vote
assert majority_vote([18.0, 20.0, 18.0, None, 18.0]) == 18.0
assert majority_vote([None, None, 5.0]) == 5.0, "None should be ignored"
assert majority_vote([None, None]) is None
print("Looks good ✅")

**7.2 Run self-consistency** on the first 10 test problems: 5 zero-shot CoT samples per problem at temperature 1.0 (50 requests).

In [ ]:
N_SC_PROBLEMS = 10
N_SAMPLES = 5

rows = []
for i, problem in enumerate(test_problems[:N_SC_PROBLEMS]):
    messages = build_messages(problem["question"], [], cot=True)
    answers = [extract_answer(chat(messages, temperature=1.0, sample=s)[0]) for s in range(N_SAMPLES)]
    vote = majority_vote(answers)
    rows.append({
        "problem": i, "gold": problem["answer"], "samples": answers, "vote": vote,
        "single sample correct": is_correct(answers[0], problem["answer"]),
        "vote correct": is_correct(vote, problem["answer"]),
        "greedy correct": results["CoT, 0-shot"]["correct"][i],
    })

sc = pd.DataFrame(rows)
print(sc[["problem", "gold", "samples", "vote"]].to_string(index=False))
print()
print(sc[["greedy correct", "single sample correct", "vote correct"]].mean())

**7.3 What do you see?** On which problems do the samples disagree, and how does that relate to whether the greedy answer was right? Self-consistency with 5 samples costs five times as much as a single answer. When is that worth it?

---

*Your answer here:*

---

### Going further (optional)

If you have rate limit left, try some of these and see how the accuracy and the token counts change:
- **Change the prompt**: give the model a persona, ask it to check its answer before the last line, or ask for a different format (for example JSON with `response_format={"type": "json_object"}`).
- **Change the demonstrations**: 1, 2 or 10 of them, or demonstrations chosen to be similar to the question.
- **Change the model**: compare a smaller and a larger model from [Groq's model list](https://console.groq.com/docs/models). Does chain-of-thought help small models more or less?
- **Built-in reasoning**: call the API with `reasoning_effort="default"` instead of `"none"` and a direct-answer prompt. The model now reasons in hidden tokens before it answers. Compare accuracy and `response.usage` with your chain-of-thought prompts.

## 8. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 8.1. In-context learning

A model is given five solved examples of a task in its prompt and then solves a sixth one correctly. What happened to the model's weights?

A. They were updated with one gradient step per example<br>
B. Only the weights of the last layer were updated<br>
C. Nothing: the examples only influence the computation through the input<br>
D. They were temporarily replaced by those of a fine-tuned model<br>

**Answer:** 

---

### 8.2. Zero-shot and few-shot

What is the difference between a zero-shot and a few-shot prompt?

A. A zero-shot prompt uses temperature 0, a few-shot prompt a higher temperature<br>
B. A few-shot prompt contains a few worked examples of the task; a zero-shot prompt only describes the task<br>
C. A few-shot prompt is sent several times and the answers are combined<br>
D. A zero-shot prompt has no system message<br>

**Answer:** 

---

### 8.3. Chain-of-thought

Why does chain-of-thought prompting improve accuracy on multi-step math problems?

A. It switches the model to a larger version behind the API<br>
B. The intermediate steps become part of the input for later tokens, so the model can build on its own intermediate results<br>
C. It makes the model's replies shorter and therefore less error-prone<br>
D. It lowers the temperature automatically<br>

**Answer:** 

---

### 8.4. Temperature

You evaluate two prompts on a benchmark and want a fair, reproducible comparison. Which temperature should you use?

A. 0, so that the model takes its most likely reply and the results do not depend on random sampling<br>
B. 1.0, because that is the model's "natural" distribution<br>
C. 2.0, to make sure the model explores enough<br>
D. It does not matter; the temperature has no effect on accuracy<br>

**Answer:** 

---

### 8.5. Self-consistency

What does self-consistency do?

A. It asks the model whether its answer is consistent with the question<br>
B. It samples several reasoning paths and takes a majority vote over the final answers<br>
C. It fine-tunes the model on its own correct answers<br>
D. It repeats the prompt until the model gives the same answer twice at temperature 0<br>

**Answer:** 